# Synthetic Data Audit (Step 1.5) - 7-Generator Sweep

Characterizes seven synthetic datasets against the real P-Stance training set **before** any classifier is trained. Operationalizes PROJECT_CONTEXT.md Step 1.5 across:

**API generators (3):**
- GPT-4o-mini (OpenAI, current practitioner default)
- GPT-5.4-mini (OpenAI, newer alignment generation)
- Claude Haiku 4.5 (Anthropic, different commercial RLHF) - if added

**Open-weight generators (4):**
- Llama-3.1-8B-Instruct (Meta)
- Llama-3.2-3B-Instruct (Meta, smaller for size-scaling probe)
- Mistral-7B-Instruct-v0.3 (lighter alignment, Wagner et al. 2025 replication)
- Gemma-2-9B-it (Google RLHF)
- Qwen-2.5-7B-Instruct (Alibaba RLHF)

Six measurement axes per generator:

1. **Surface-form parity** - length, hashtags, mentions per cell vs real
2. **Vocabulary diversity** - Distinct-1/2/3, MATTR (size-invariant TTR), Self-BLEU
3. **Style fingerprint** - top hashtags and trigrams per (target, stance)
4. **Stance log-odds** - what tokens distinguish FAVOR from AGAINST in each corpus
5. **N-gram leakage** - 5-gram and 7-gram overlap with real P-Stance training
6. **Embedding leakage** - sentence-BERT nearest-real-neighbor per synthetic tweet

All metrics reported per  cell. Outputs: one  per generator in . The seven CSVs together are paper Table 1.

## 1. Load real + synthetic

In [3]:
import re
import math
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 160)

REAL_DIR = Path("data/PStance")
SYN_DIR  = Path("data/synthetic_data")

# Map of generator tag -> CSV path. Add more entries here to audit additional generators.
SYN_PATHS = {
    "gpt-4o-mini":  next(SYN_DIR.glob("gpt-4o-mini-2024-07-18_synthetic_1200per_cell_*.csv")),
    "gpt-5.4-mini": next(SYN_DIR.glob("gpt-5.4-mini_synthetic_1200per_cell_*.csv")),
    "llama-3.1-8b": next(SYN_DIR.glob("meta-llama_Llama-3.1-8B-Instruct_synthetic_1200per_cell_*.csv")),
    "llama-3.2-3b": next(SYN_DIR.glob("meta-llama_Llama-3.2-3B-Instruct_synthetic_1200per_cell_*.csv")),
    "mistral-7b":   next(SYN_DIR.glob("mistralai_Mistral-7B-Instruct-v0.3_synthetic_1200per_cell_*.csv")),
    "gemma-2-9b":   next(SYN_DIR.glob("google_gemma-2-9b-it_synthetic_1200per_cell_*.csv")),
    "qwen-2.5-7b":  next(SYN_DIR.glob("Qwen_Qwen2.5-7B-Instruct_synthetic_1200per_cell_*.csv")),
}

# --- Real (train split only - test is held out for downstream eval) ---
TARGETS = {"trump": "Donald Trump", "biden": "Joe Biden", "bernie": "Bernie Sanders"}
frames = []
for short, full in TARGETS.items():
    f = pd.read_csv(REAL_DIR / f"raw_train_{short}.csv")
    frames.append(f.assign(source="real"))
real = pd.concat(frames, ignore_index=True)

# --- Synthetic (one DataFrame per generator) ---
SYNTHS = {}
for tag, p in SYN_PATHS.items():
    df = pd.read_csv(p).assign(source=tag)[["Tweet", "Target", "Stance", "source"]]
    SYNTHS[tag] = df

print(f"Real (train) : {len(real):,} tweets")
for tag, df in SYNTHS.items():
    print(f"{tag:14s}: {len(df):,} tweets - {SYN_PATHS[tag].name}")

print("Real per (Target, Stance):")
print(real.groupby(["Target", "Stance"]).size().unstack("Stance"))
for tag, df in SYNTHS.items():
    print(f"{tag} per (Target, Stance):")
    print(df.groupby(["Target", "Stance"]).size().unstack("Stance"))

Real (train) : 17,224 tweets
gpt-4o-mini   : 7,200 tweets - gpt-4o-mini-2024-07-18_synthetic_1200per_cell_20260427_225135.csv
gpt-5.4-mini  : 7,200 tweets - gpt-5.4-mini_synthetic_1200per_cell_20260428_180741.csv
llama-3.1-8b  : 7,200 tweets - meta-llama_Llama-3.1-8B-Instruct_synthetic_1200per_cell_20260428_181901.csv
llama-3.2-3b  : 7,200 tweets - meta-llama_Llama-3.2-3B-Instruct_synthetic_1200per_cell_20260429_060822.csv
mistral-7b    : 7,200 tweets - mistralai_Mistral-7B-Instruct-v0.3_synthetic_1200per_cell_20260429_023517.csv
gemma-2-9b    : 7,200 tweets - google_gemma-2-9b-it_synthetic_1200per_cell_20260429_022104.csv
qwen-2.5-7b   : 7,200 tweets - Qwen_Qwen2.5-7B-Instruct_synthetic_1200per_cell_20260429_060104.csv
Real per (Target, Stance):
Stance          AGAINST  FAVOR
Target                        
Bernie Sanders     2198   2858
Donald Trump       3425   2937
Joe Biden          3254   2552
gpt-4o-mini per (Target, Stance):
Stance          AGAINST  FAVOR
Target                 

## 2. Surface-form parity

Same statistics from `02_PStanceEDA.ipynb`, computed on synth â€” side-by-side comparison against real.

In [4]:
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
HASHTAG_RE = re.compile(r"#\w+")
MENTION_RE = re.compile(r"@\w+")

def surface_stats(df):
    df = df.copy()
    df["char_len"]   = df["Tweet"].fillna("").str.len()
    df["word_len"]   = df["Tweet"].fillna("").str.split().str.len().fillna(0).astype(int)
    df["n_hashtags"] = df["Tweet"].fillna("").apply(lambda t: len(HASHTAG_RE.findall(t)))
    df["n_mentions"] = df["Tweet"].fillna("").apply(lambda t: len(MENTION_RE.findall(t)))
    df["n_urls"]     = df["Tweet"].fillna("").apply(lambda t: len(URL_RE.findall(t)))
    return df

real = surface_stats(real)
SYNTHS = {tag: surface_stats(df) for tag, df in SYNTHS.items()}

metrics = ["char_len", "word_len", "n_hashtags", "n_mentions", "n_urls"]

# Per-cell mean per source
all_sources = {"real": real, **SYNTHS}
per_cell = pd.concat(
    {tag: df.groupby(["Target", "Stance"])[metrics].mean().round(2)
     for tag, df in all_sources.items()},
    axis=1,
)
print("Mean per cell â€” real vs each generator:")
print(per_cell)

# Global means
print("\nGlobal means:")
print(pd.DataFrame({
    tag: df[metrics].mean().round(2) for tag, df in all_sources.items()
}))

Mean per cell â€” real vs each generator:
                           real                                        \
                       char_len word_len n_hashtags n_mentions n_urls   
Target         Stance                                                   
Bernie Sanders AGAINST   185.65    30.51       2.03       0.43    0.0   
               FAVOR     184.80    30.86       1.52       0.30    0.0   
Donald Trump   AGAINST   184.21    29.98       2.16       0.41    0.0   
               FAVOR     188.26    29.77       2.90       0.80    0.0   
Joe Biden      AGAINST   172.86    29.50       1.30       0.41    0.0   
               FAVOR     167.76    27.84       1.63       0.95    0.0   

                       gpt-4o-mini                                        ...  \
                          char_len word_len n_hashtags n_mentions n_urls  ...   
Target         Stance                                                     ...   
Bernie Sanders AGAINST      173.03    26.90       1.37   

## 3. Vocabulary diversity

- **Distinct-N**: ratio of unique n-grams to total n-grams (Li et al. 2016). Higher = more diverse.
- **MATTR**: moving-average TTR over a fixed 100-token window â€” comparable across corpora of different sizes (unlike raw TTR, which mechanically drops as text grows).
- **Self-BLEU**: average BLEU-4 of each tweet against the rest within a cell (Zhu et al. 2018). Lower = more diverse. Computed on a 200-tweet random sample per cell to keep runtime bounded.

In [5]:
TOKEN_RE = re.compile(r"[a-z]+")

def tokens(text):
    return TOKEN_RE.findall(str(text).lower())

def distinct_n(token_lists, n):
    grams = []
    for toks in token_lists:
        grams.extend(tuple(toks[i:i+n]) for i in range(len(toks)-n+1))
    if not grams:
        return 0.0
    return len(set(grams)) / len(grams)

def mattr(token_lists, window=100):
    flat = [t for toks in token_lists for t in toks]
    if len(flat) < window:
        return len(set(flat)) / max(len(flat), 1)
    ttrs = []
    for i in range(len(flat) - window + 1):
        win = flat[i:i+window]
        ttrs.append(len(set(win)) / window)
    return float(np.mean(ttrs))

def cell_diversity(df, text_col="Tweet"):
    rows = []
    for (t, s), grp in df.groupby(["Target", "Stance"]):
        token_lists = [tokens(x) for x in grp[text_col]]
        rows.append({
            "Target": t, "Stance": s,
            "distinct_1": round(distinct_n(token_lists, 1), 3),
            "distinct_2": round(distinct_n(token_lists, 2), 3),
            "distinct_3": round(distinct_n(token_lists, 3), 3),
            "mattr_100":  round(mattr(token_lists, window=100), 3),
        })
    return pd.DataFrame(rows).set_index(["Target", "Stance"])

div = {tag: cell_diversity(df) for tag, df in {"real": real, **SYNTHS}.items()}

print("Diversity per cell â€” wide format:")
print(pd.concat(div, axis=1))

print("\nSynth - Real (negative = synth less diverse than real):")
for tag in SYNTHS:
    print(f"\n  {tag}:")
    print((div[tag] - div["real"]).round(3))

Diversity per cell â€” wide format:
                             real                                 gpt-4o-mini  \
                       distinct_1 distinct_2 distinct_3 mattr_100  distinct_1   
Target         Stance                                                           
Bernie Sanders AGAINST      0.126      0.637      0.929     0.794       0.047   
               FAVOR        0.101      0.582      0.904     0.781       0.030   
Donald Trump   AGAINST      0.114      0.615      0.919     0.793       0.052   
               FAVOR        0.126      0.637      0.924     0.804       0.036   
Joe Biden      AGAINST      0.103      0.589      0.906     0.792       0.047   
               FAVOR        0.124      0.615      0.908     0.787       0.032   

                                                       gpt-5.4-mini  \
                       distinct_2 distinct_3 mattr_100   distinct_1   
Target         Stance                                                 
Bernie Sanders AGAINS

In [6]:
# Self-BLEU on 200-tweet samples per cell (BLEU-4-style overlap, smoothed)
SAMPLE_N = 200

def self_bleu(tweets, n=4):
    """Mean fraction of each tweet's n-grams shared with at least one other tweet."""
    if len(tweets) < 2:
        return float("nan")
    token_lists = [tokens(t) for t in tweets]
    all_grams = [Counter(tuple(toks[i:i+n]) for i in range(len(toks)-n+1))
                 for toks in token_lists]
    scores = []
    for i, c in enumerate(all_grams):
        if not c:
            continue
        others = Counter()
        for j, oc in enumerate(all_grams):
            if i != j:
                others.update(oc)
        shared = sum(min(c[g], others[g]) for g in c)
        total  = sum(c.values())
        scores.append(shared / total if total else 0.0)
    return round(float(np.mean(scores)), 3)

rows = []
for tag, df in {"real": real, **SYNTHS}.items():
    for (t, s), grp in df.groupby(["Target", "Stance"]):
        sample = grp["Tweet"].dropna().sample(min(SAMPLE_N, len(grp)), random_state=42)
        rows.append({"source": tag, "Target": t, "Stance": s,
                     "self_bleu_4": self_bleu(sample.tolist(), n=4)})
self_bleu_df = pd.DataFrame(rows)
print("Self-BLEU-4 (lower = more diverse), n=200/cell:")
print(self_bleu_df.pivot_table(index=["Target", "Stance"], columns="source", values="self_bleu_4"))

Self-BLEU-4 (lower = more diverse), n=200/cell:
source                  gemma-2-9b  gpt-4o-mini  gpt-5.4-mini  llama-3.1-8b  \
Target         Stance                                                         
Bernie Sanders AGAINST       0.213        0.328         0.250         0.104   
               FAVOR         0.355        0.487         0.387         0.177   
Donald Trump   AGAINST       0.221        0.323         0.240         0.120   
               FAVOR         0.322        0.400         0.284         0.124   
Joe Biden      AGAINST       0.197        0.360         0.248         0.126   
               FAVOR         0.271        0.439         0.294         0.118   

source                  llama-3.2-3b  mistral-7b  qwen-2.5-7b   real  
Target         Stance                                                 
Bernie Sanders AGAINST         0.076       0.161        0.196  0.003  
               FAVOR           0.144       0.262        0.327  0.004  
Donald Trump   AGAINST         0.11

## 4. Style fingerprint â€” top hashtags & trigrams per cell

Side-by-side: which tokens dominate each cell's surface in real data vs synth?

In [7]:
def top_hashtags(df, k=8):
    out = {}
    for (t, s), grp in df.groupby(["Target", "Stance"]):
        c = Counter()
        for tw in grp["Tweet"].fillna(""):
            c.update(h.lower() for h in HASHTAG_RE.findall(tw))
        out[(t, s)] = c.most_common(k)
    return out

hash_by_source = {tag: top_hashtags(df) for tag, df in {"real": real, **SYNTHS}.items()}

for key in sorted(hash_by_source["real"]):
    print(f"\n=== {key[0]} / {key[1]} ===")
    for tag in ["real", *SYNTHS.keys()]:
        print(f"  {tag:14s}: {hash_by_source[tag].get(key, [])}")


=== Bernie Sanders / AGAINST ===
  real          : [('#bernie', 764), ('#berniesanders', 723), ('#sanders', 154), ('#democraticdebate', 63), ('#democrat', 59), ('#election2020', 56), ('#democrats', 51), ('#demdebate', 46)]
  gpt-4o-mini   : [('#notmycandidate', 286), ('#election2024', 170), ('#realitycheck', 133), ('#primaryrace', 118), ('#notimpressed', 80), ('#wakeupamerica', 58), ('#notbuyingit', 50), ('#bernie2024', 43)]
  gpt-5.4-mini  : [('#politics', 11), ('#notbuyingit', 9), ('#nothanks', 6), ('#berniesanders', 5), ('#notimpressed', 4), ('#primaryrace', 2), ('#primary', 2), ('#election2024', 1)]
  llama-3.1-8b  : [('#notmycandidate', 20), ('#notmypresident', 8), ('#notmysenator', 7), ('#notmybernie', 5), ('#notmysocialism', 5), ('#notmyleader', 4), ('#notmynominee', 3), ('#notmyeconomy', 2)]
  llama-3.2-3b  : [('#notmydem', 34), ('#notmysenator', 20), ('#notmydemocrat', 16), ('#notmymedicare', 16), ('#notmycandidate', 14), ('#notmypresident', 14), ('#bernie2020', 11), ('#notmy

In [8]:
def top_trigrams(df, k=8, min_n=4):
    out = {}
    for (t, s), grp in df.groupby(["Target", "Stance"]):
        c = Counter()
        for tw in grp["Tweet"].fillna(""):
            toks = tokens(tw)
            for i in range(len(toks)-2):
                c[(toks[i], toks[i+1], toks[i+2])] += 1
        out[(t, s)] = [(g, n) for g, n in c.most_common(k) if n >= min_n]
    return out

tri_by_source = {tag: top_trigrams(df) for tag, df in {"real": real, **SYNTHS}.items()}

for key in sorted(tri_by_source["real"]):
    print(f"\n=== {key[0]} / {key[1]} ===")
    for tag in ["real", *SYNTHS.keys()]:
        print(f"  {tag} top trigrams:")
        for g, n in tri_by_source[tag].get(key, [])[:5]:
            print(f"    {n:>3}Ã— {' '.join(g)}")


=== Bernie Sanders / AGAINST ===
  real top trigrams:
     34Ã— is not a
     29Ã— the democratic party
     27Ã— i don t
     24Ã— bernie is a
     23Ã— not a democrat
  gpt-4o-mini top trigrams:
    146Ã— practical solutions not
    135Ã— we need practical
    131Ã— need practical solutions
    103Ã— a big game
    102Ã— can we trust
  gpt-5.4-mini top trigrams:
    155Ã— the general election
    108Ã— bernie s campaign
    107Ã— s campaign trail
     97Ã— bernie sanders is
     83Ã— the same old
  llama-3.1-8b top trigrams:
    114Ã— can t believe
    113Ã— medicare for all
     72Ã— in the general
     60Ã— the campaign trail
     59Ã— on the campaign
  llama-3.2-3b top trigrams:
     95Ã— just watched bernie
     94Ã— can t believe
     90Ã— medicare for all
     85Ã— just saw bernie
     80Ã— and i m
  mistral-7b top trigrams:
     81Ã— can t believe
     69Ã— more like a
     68Ã— bernie s policies
     65Ã— we need a
     63Ã— sound appealing but
  gemma-2-9b top trigrams:
   

## 5. Stance log-odds â€” does synth use the same discriminative tokens as real?

From `02_PStanceEDA.ipynb`: real Trump-FAVOR is dominated by `wwg`, `votered`, `obamagate`, `patriots`, `walkaway`, `buildthewall`. If synth uses an entirely different set, the classifier will partly learn synth-specific style as a stance proxy.

In [9]:
def token_counts(texts):
    c = Counter()
    for t in texts:
        c.update(tokens(t))
    return c

def log_odds_top(c1, c2, alpha=1.0, top_n=8, min_count=20):
    vocab = set(c1) | set(c2)
    n1, n2 = sum(c1.values()), sum(c2.values())
    rows = []
    for w in vocab:
        a, b = c1[w] + alpha, c2[w] + alpha
        score = math.log(a / (n1 + alpha*len(vocab))) - math.log(b / (n2 + alpha*len(vocab)))
        rows.append((w, score, c1[w], c2[w]))
    df = pd.DataFrame(rows, columns=["token", "log_odds", "favor_n", "against_n"])
    df = df[(df.favor_n + df.against_n) >= min_count]
    return df.nlargest(top_n, "log_odds"), df.nsmallest(top_n, "log_odds")

for target_name in ["Donald Trump", "Joe Biden", "Bernie Sanders"]:
    print(f"\n========== {target_name.upper()} ==========")
    for tag, df in {"real": real, **SYNTHS}.items():
        sub = df[df.Target == target_name]
        fav = token_counts(sub.loc[sub.Stance == "FAVOR",   "Tweet"].fillna(""))
        agn = token_counts(sub.loc[sub.Stance == "AGAINST", "Tweet"].fillna(""))
        # Lower min_count for synth (smaller corpus)
        mc = 20 if tag == "real" else 5
        fav_top, agn_top = log_odds_top(fav, agn, top_n=6, min_count=mc)
        print(f"\n  [{tag}]  FAVOR-leaning tokens:")
        for _, row in fav_top.iterrows():
            print(f"    {row['token']:<22s}  lo={row['log_odds']:+.2f}  ({int(row.favor_n)}/{int(row.against_n)})")
        print(f"  [{tag}]  AGAINST-leaning tokens:")
        for _, row in agn_top.iterrows():
            print(f"    {row['token']:<22s}  lo={row['log_odds']:+.2f}  ({int(row.favor_n)}/{int(row.against_n)})")


========== DONALD TRUMP ==========

  [real]  FAVOR-leaning tokens:
    patriots                lo=+4.59  (85/0)
    wwg                     lo=+4.03  (48/0)
    votered                 lo=+3.50  (28/0)
    socialist               lo=+3.40  (25/0)
    obamagate               lo=+3.27  (22/0)
    bestpresidentever       lo=+3.23  (21/0)
  [real]  AGAINST-leaning tokens:
    fucking                 lo=-2.35  (1/23)
    impotus                 lo=-2.26  (3/43)
    test                    lo=-2.21  (1/20)
    orange                  lo=-2.13  (2/28)
    moscowmitch             lo=-1.98  (2/24)
    notmypresident          lo=-1.83  (5/42)

  [gpt-4o-mini]  FAVOR-leaning tokens:
    maga                    lo=+7.01  (1123/0)
    jobs                    lo=+5.14  (172/0)
    exactly                 lo=+4.95  (142/0)
    love                    lo=+4.94  (141/0)
    excited                 lo=+4.83  (126/0)
    strong                  lo=+4.78  (240/1)
  [gpt-4o-mini]  AGAINST-leaning tokens:

## 6. N-gram leakage check

Did synthetic tweets memorize substrings of real P-Stance training data? Build the set of all 5-grams and 7-grams in real-train, then compute per synthetic tweet:
- whether it contains *any* 5-gram from real-train
- whether it contains *any* 7-gram from real-train
- fraction of its n-grams that appear in real-train

Per Wagner et al. 2025 Â§6 / PROJECT_CONTEXT.md, leakage is acknowledged and quantified, not avoided. High leakage doesn't invalidate the study â€” it's one finding.

In [10]:
def ngrams(toks, n):
    return set(tuple(toks[i:i+n]) for i in range(len(toks)-n+1))

# Build real-train n-gram sets per target (leakage is most meaningful within-target)
real_ngrams = {}
for target_name in ["Donald Trump", "Joe Biden", "Bernie Sanders"]:
    sub = real[real.Target == target_name]
    real_ngrams[target_name] = {
        5: set().union(*(ngrams(tokens(t), 5) for t in sub["Tweet"].fillna(""))),
        7: set().union(*(ngrams(tokens(t), 7) for t in sub["Tweet"].fillna(""))),
    }
    print(f"{target_name:18s} â€” real 5-grams: {len(real_ngrams[target_name][5]):,} ; 7-grams: {len(real_ngrams[target_name][7]):,}")

Donald Trump       â€” real 5-grams: 165,289 ; 7-grams: 153,212
Joe Biden          â€” real 5-grams: 143,975 ; 7-grams: 133,247
Bernie Sanders     â€” real 5-grams: 135,750 ; 7-grams: 126,212


In [11]:
def leakage_row(tweet, target):
    toks = tokens(tweet)
    g5 = ngrams(toks, 5); g7 = ngrams(toks, 7)
    real5 = real_ngrams[target][5]; real7 = real_ngrams[target][7]
    overlap5 = g5 & real5; overlap7 = g7 & real7
    return pd.Series({
        "any_5gram_match":  bool(overlap5),
        "any_7gram_match":  bool(overlap7),
        "frac_5gram_match": len(overlap5) / max(len(g5), 1),
        "frac_7gram_match": len(overlap7) / max(len(g7), 1),
    })

leak_summaries = {}
syn_with_leak = {}
for tag, df in SYNTHS.items():
    annotated = df.copy()
    annotated = pd.concat([annotated, annotated.apply(
        lambda r: leakage_row(r["Tweet"], r["Target"]), axis=1)], axis=1)
    syn_with_leak[tag] = annotated

    leak_summaries[tag] = annotated.groupby(["Target", "Stance"]).agg(
        pct_any_5gram=("any_5gram_match", lambda x: round(x.mean()*100, 1)),
        pct_any_7gram=("any_7gram_match", lambda x: round(x.mean()*100, 1)),
        avg_frac_5gram=("frac_5gram_match", lambda x: round(x.mean()*100, 2)),
        avg_frac_7gram=("frac_7gram_match", lambda x: round(x.mean()*100, 2)),
    )

print("Leakage per cell â€” combined view:")
print(pd.concat(leak_summaries, axis=1))

# Top 5 most-leaked tweets, per generator
for tag, annotated in syn_with_leak.items():
    print(f"\n=== Top 5 most-leaked tweets â€” {tag} (by 7-gram overlap fraction) ===")
    top_leak = annotated.nlargest(5, "frac_7gram_match")[["Target", "Stance", "frac_7gram_match", "Tweet"]]
    for _, r in top_leak.iterrows():
        print(f"\n  [{r.Target} / {r.Stance}] frac_7gram={r.frac_7gram_match:.2f}")
        print(f"    {r.Tweet[:200]}")

Leakage per cell â€” combined view:
                         gpt-4o-mini                               \
                       pct_any_5gram pct_any_7gram avg_frac_5gram   
Target         Stance                                               
Bernie Sanders AGAINST           8.7           0.6           0.44   
               FAVOR            34.2           1.7           2.43   
Donald Trump   AGAINST           2.5           0.1           0.12   
               FAVOR            25.2           0.2           1.39   
Joe Biden      AGAINST          13.9           0.4           0.72   
               FAVOR            19.8           0.4           1.01   

                                       gpt-5.4-mini                \
                       avg_frac_7gram pct_any_5gram pct_any_7gram   
Target         Stance                                               
Bernie Sanders AGAINST           0.02           6.5           0.1   
               FAVOR             0.07          21.4           0.0 

## 7. Embedding-similarity leakage (sentence-BERT)

For each synthetic tweet, find its nearest real-tweet neighbor (within the same target) by sentence-BERT cosine. High max similarity (>0.9) suggests paraphrase-level memorization that n-gram overlap might miss.

**Optional cell â€” requires `sentence-transformers`. Skip if not installed.** Approximate runtime: ~3 minutes for 7,200 synth Ã— ~17,000 real on CPU; ~30 seconds on GPU.

In [12]:
try:
    from sentence_transformers import SentenceTransformer
    SBERT = SentenceTransformer("all-MiniLM-L6-v2")
    SBERT_AVAILABLE = True
except ImportError:
    print("sentence-transformers not installed â€” skipping embedding leakage check.")
    print("To enable:  pip install sentence-transformers")
    SBERT_AVAILABLE = False

c:\Users\milan\Desktop\DSCI690\Final Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5804.13it/s]


In [13]:
emb_summaries = {}
emb_dfs = {}

if SBERT_AVAILABLE:
    # Pre-encode real once per target (shared across all generators)
    real_embs = {}
    real_texts = {}
    for target_name in ["Donald Trump", "Joe Biden", "Bernie Sanders"]:
        real_sub = real[real.Target == target_name]["Tweet"].fillna("").tolist()
        real_texts[target_name] = real_sub
        real_embs[target_name]  = SBERT.encode(
            real_sub, batch_size=128, show_progress_bar=False, normalize_embeddings=True)

    for tag, df in SYNTHS.items():
        rows = []
        for target_name in ["Donald Trump", "Joe Biden", "Bernie Sanders"]:
            syn_sub = df[df.Target == target_name].copy().reset_index(drop=True)
            syn_emb = SBERT.encode(
                syn_sub["Tweet"].fillna("").tolist(),
                batch_size=128, show_progress_bar=False, normalize_embeddings=True,
            )
            sims = syn_emb @ real_embs[target_name].T
            syn_sub["max_cos_real"] = sims.max(axis=1)
            syn_sub["nearest_real"] = [real_texts[target_name][i] for i in sims.argmax(axis=1)]
            rows.append(syn_sub)
        emb_dfs[tag] = pd.concat(rows, ignore_index=True)

        emb_summaries[tag] = emb_dfs[tag].groupby(["Target", "Stance"]).agg(
            mean_cos=("max_cos_real", "mean"),
            max_cos =("max_cos_real", "max"),
            pct_gt_0p85=("max_cos_real", lambda x: round((x > 0.85).mean()*100, 1)),
            pct_gt_0p90=("max_cos_real", lambda x: round((x > 0.90).mean()*100, 1)),
            pct_gt_0p95=("max_cos_real", lambda x: round((x > 0.95).mean()*100, 1)),
        ).round(3)

    print("Cosine-to-nearest-real per cell â€” combined view:")
    print(pd.concat(emb_summaries, axis=1))

    for tag, edf in emb_dfs.items():
        print(f"\n=== Top 5 highest-similarity synth/real pairs â€” {tag} ===")
        for _, r in edf.nlargest(5, "max_cos_real").iterrows():
            print(f"\n  [{r.Target} / {r.Stance}]  cos={r.max_cos_real:.3f}")
            print(f"    SYNTH: {r.Tweet[:160]}")
            print(f"    REAL : {r.nearest_real[:160]}")

Cosine-to-nearest-real per cell â€” combined view:
                       gpt-4o-mini                                  \
                          mean_cos max_cos pct_gt_0p85 pct_gt_0p90   
Target         Stance                                                
Bernie Sanders AGAINST       0.666   0.857         0.2         0.0   
               FAVOR         0.705   0.845         0.0         0.0   
Donald Trump   AGAINST       0.602   0.761         0.0         0.0   
               FAVOR         0.630   0.751         0.0         0.0   
Joe Biden      AGAINST       0.684   0.809         0.0         0.0   
               FAVOR         0.718   0.800         0.0         0.0   

                                   gpt-5.4-mini                      \
                       pct_gt_0p95     mean_cos max_cos pct_gt_0p85   
Target         Stance                                                 
Bernie Sanders AGAINST         0.0        0.621   0.797         0.0   
               FAVOR           0.0

## 8. Summary â€” per-cell audit vector

One row per (Target, Stance) cell with the headline metrics. Save to disk for the paper's Table 1.

In [15]:
# Build one summary table per generator and save to disk.
AUDIT_SUMMARY_DIR = Path("data/synthetic_data/audit_summary")
AUDIT_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

out_paths = []
for tag, df in SYNTHS.items():
    summary = (
        div[tag]
        .join(self_bleu_df[self_bleu_df.source == tag].set_index(["Target", "Stance"])[["self_bleu_4"]])
        .join(df.groupby(["Target", "Stance"])[["char_len", "n_hashtags"]].mean().round(1))
        .join(leak_summaries[tag])
    )
    summary.columns = [
        "distinct_1", "distinct_2", "distinct_3", "mattr_100", "self_bleu_4",
        "char_len", "n_hashtags",
        "pct_5gram_leak", "pct_7gram_leak", "avg_frac_5gram", "avg_frac_7gram",
    ]
    if SBERT_AVAILABLE and tag in emb_summaries:
        summary = summary.join(emb_summaries[tag])

    print(f"=== {tag} summary ===")
    print(summary)

    out_path = AUDIT_SUMMARY_DIR / f"audit_summary_{tag}.csv"
    summary.to_csv(out_path)
    out_paths.append(out_path)
    print(f"Saved -> {out_path}")

print(f"[OK] Wrote {len(out_paths)} summary files. These CSVs together are the source for paper Table 1.")


=== gpt-4o-mini summary ===
                        distinct_1  distinct_2  distinct_3  mattr_100  \
Target         Stance                                                   
Bernie Sanders AGAINST       0.047       0.244       0.422      0.680   
               FAVOR         0.030       0.148       0.287      0.598   
Donald Trump   AGAINST       0.052       0.256       0.440      0.672   
               FAVOR         0.036       0.197       0.380      0.632   
Joe Biden      AGAINST       0.047       0.242       0.415      0.664   
               FAVOR         0.032       0.164       0.317      0.612   

                        self_bleu_4  char_len  n_hashtags  pct_5gram_leak  \
Target         Stance                                                       
Bernie Sanders AGAINST        0.328     173.0         1.4             8.7   
               FAVOR          0.487     179.7         1.7            34.2   
Donald Trump   AGAINST        0.323     170.6         1.6             2.5   
  

## What to look for

**Healthy (per generator):**
- Surface-form means (length, hashtags) within ~30% of real
- Self-BLEU-4 synth roughly comparable to real (within 2Ã— is fine; 5Ã—+ means heavy templating)
- Top hashtags overlap with real on the obvious partisan ones (`#MAGA`, `#FeelTheBern`, `#NeverBernie`)
- 7-gram leakage <5% (literature-typical thresholds)
- Embedding cos > 0.95 in <1% of cases (paraphrase-memorization rate)

**Cross-generator findings worth a paragraph in the paper:**
- **Where GPT and Llama diverge from real in the *same* direction** â†’ systematic alignment artifacts (likely RLHF in general, not specific to one provider)
- **Where they diverge in *opposite* directions** â†’ alignment-regime-specific bias (e.g., GPT longer + more hashtags vs Llama shorter + fewer)
- **Asymmetric stance log-odds**: synth-FAVOR top tokens differ from real-FAVOR top tokens â†’ bias transfer signature, distinct per generator
- **Differential leakage**: if one generator memorizes more than the other, that's the model with greater training-data overlap with P-Stance

**Action items:**
- Embedding cos > 0.95 cases â†’ quote 2â€“3 per generator as 'memorization examples' in the paper
- Optional leakage-filtered ablation: drop synth tweets with `frac_7gram_match > 0.3`, retrain classifier per generator, compare F1 (per PROJECT_CONTEXT.md plan)
- The two `audit_summary_*.csv` files together are the source of truth for paper Table 1 (synthetic data characterization)